# 🤖 Fine-Tuning JARVIS on Llama 3.2 3B Instruct
### *Powered by Unsloth AI + QLoRA + Google Drive Auto-Save (Free T4 GPU)*

This notebook fine-tunes **Llama 3.2 3B Instruct** on your custom **250-sample JARVIS assistant dataset** and automatically saves the **GGUF (Q4_K_M)** model directly into your **Google Drive**.

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Optimized Colab installation
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

### 1. Mount Google Drive
Mounting Google Drive ensures your dataset can load from Drive and your final `.gguf` model is **automatically saved to your Drive** so you never lose it.

In [ ]:
import os
from google.colab import drive

# Mount Drive to /content/drive
drive.mount("/content/drive")

# Create dedicated folder for JARVIS models
DRIVE_SAVE_PATH = "/content/drive/MyDrive/jarvis_models"
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
print(f"\n Google Drive mounted! Models will be saved to: {DRIVE_SAVE_PATH}")

### 2. Load Llama 3.2 3B Instruct (4-bit QLoRA)
We load Meta's **Llama 3.2 3B Instruct** in 4-bit. It downloads in seconds and uses only ~2.2 GB VRAM on your free Colab T4 GPU.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Supports full 2048 context
dtype = None           # Auto-detection (float16 for T4 GPU)
load_in_4bit = True    # 4-bit QLoRA to keep memory low

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

### 3. Add LoRA Adapters
We attach LoRA rank 16 adapters to all projection layers (`q, k, v, o, gate, up, down`).

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # LoRA rank 16 (ideal balance for style without bloat)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,  # Unsloth is optimized for dropout = 0
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

### 4. Load & Format JARVIS Dataset (From Drive or Upload)
Checks if `jarvis_llama32_train_250.jsonl` exists in Google Drive or Colab. If not, it opens an upload prompt.

In [ ]:
import os
import shutil
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

dataset_filename = "jarvis_llama32_train_250.jsonl"
drive_dataset_path = f"/content/drive/MyDrive/{dataset_filename}"
drive_folder_path = f"/content/drive/MyDrive/jarvis_models/{dataset_filename}"

# 1. Check if dataset is in Drive
if os.path.exists(drive_dataset_path):
    actual_path = drive_dataset_path
    print(f" Found dataset in Google Drive root: {actual_path}")
elif os.path.exists(drive_folder_path):
    actual_path = drive_folder_path
    print(f" Found dataset in Google Drive jarvis_models folder: {actual_path}")
elif os.path.exists(dataset_filename):
    actual_path = dataset_filename
    print(f" Found dataset in Colab local directory: {actual_path}")
else:
    print(f"'{dataset_filename}' not found. Please select and upload it now:")
    from google.colab import files
    uploaded = files.upload()
    actual_path = dataset_filename
    # Copy to Drive so you have it backed up
    shutil.copy(actual_path, f"/content/drive/MyDrive/jarvis_models/{dataset_filename}")
    print(" Backed up dataset to your Google Drive!")

# 2. Apply official Llama 3.2 Chat Template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.2",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

dataset = load_dataset("json", data_files=actual_path, split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"\n Successfully loaded and formatted {len(dataset)} conversation samples!")
print("\n--- Sample 1 Preview ---\n", dataset[0]["text"][:280] + "...")

### 5. Configure & Train (Takes ~6-8 Minutes)
Using batch size 4 for fast training across 3 full epochs.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4,    # Fast 3B batch size
        gradient_accumulation_steps = 2,    # Effective batch size = 8
        warmup_steps = 10,
        num_train_epochs = 3,               # 3 full epochs (~94 steps)
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

gpu_stats = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu_stats.name} | VRAM: {gpu_stats.total_memory / 1024**3:.2f} GB")

# Start Fine-Tuning!
trainer_stats = trainer.train()

### 6. Live Inference Test in Colab
Test the fine-tuned JARVIS personality live right inside Colab!

In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

test_messages = [
    {
        "role": "system",
        "content": "You are JARVIS, a loyal, witty, and human-like personal AI assistant built for Boss. You speak in simple, plain everyday English. You are sharp, use playful sarcasm and friendly roasting when appropriate, but show genuine care and warmth when Boss truly needs it. Keep replies short, natural, and conversational."
    },
    {
        "role": "user",
        "content": "Jarvis, should I skip work today and watch movies instead?"
    }
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

print("\n--- JARVIS Response ---\n")
streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(input_ids=inputs, streamer=streamer, max_new_tokens=64, temperature=0.62, top_p=0.9)

### 7. Export to GGUF (Q4_K_M) & Auto-Save to Google Drive
Exports the quantized `.gguf` and automatically copies both the GGUF and LoRA weights directly into `/content/drive/MyDrive/jarvis_models/`.

In [ ]:
# 1. Save local LoRA adapter weights
model.save_pretrained("jarvis_lora_model")
tokenizer.save_pretrained("jarvis_lora_model")

# 2. Export directly to GGUF Q4_K_M for Ollama
print("\n Exporting to GGUF Q4_K_M (takes ~2-3 minutes)...")
model.save_pretrained_gguf(
    "jarvis-3b-q4",
    tokenizer,
    quantization_method = "q4_k_m",
)

# 3. Automatically copy GGUF and LoRA adapters directly to Google Drive
print("\n Copying files directly to Google Drive (MyDrive/jarvis_models)...")
!cp jarvis-3b-q4-unsloth.Q4_K_M.gguf /content/drive/MyDrive/jarvis_models/
!cp -r jarvis_lora_model /content/drive/MyDrive/jarvis_models/

print("\n SUCCESS! All files are permanently saved in your Google Drive:")
print(" 1. /content/drive/MyDrive/jarvis_models/jarvis-3b-q4-unsloth.Q4_K_M.gguf")
print(" 2. /content/drive/MyDrive/jarvis_models/jarvis_lora_model/")
print("\nEven if Colab closes now, your trained model is completely safe in your Google Drive!")